# 99 — Pipeline status
Shows only what `state/*.json` and local artifacts prove. Set `CHECK_REMOTE=True` for a
read-only Camber listing of v6p5_ext / v6p5_compose / training tags.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
from yolov11sdi.reporting import status
cfg = load_config("yolo11n_ablation")
display(status.stage_table(cfg, paths))
ov = status.overview(cfg, paths)
display(pd.Series({k: v for k, v in ov.items() if k != "disk"}, dtype=object).to_frame("value"))
d = ov["disk"]; print(f"disk free {d['free_gb']:.1f} / {d['total_gb']:.1f} GB at {d['path']}")

In [ ]:
CHECK_REMOTE = False
if CHECK_REMOTE:
    from yolov11sdi.camber import CamberClient
    from yolov11sdi.environment import load_camber_api_key
    load_camber_api_key(paths.env)
    print(status.remote_tags(cfg, CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))))

In [ ]:
for name, st in store.all().items():
    if st.status in ("failed", "needs_review"):
        print(f"--- {name}: {st.status}"); print(st.error or st.summary)